# VOICEVOX Japanese TTS in Google Colab (Direct Linux GPU / CPU Engine)

**Goal:** Run the official VOICEVOX Engine binary directly on Linux in Google Colab with automatic NVIDIA GPU (CUDA) acceleration and CPU fallback—**zero Docker required**—with query validation, test mode, and 0.5s pre/post roll padding across presentation slides.

**Colab Runtime Recommendation:**
- **Hardware Accelerator:** `T4 GPU` (*Runtime > Change runtime type > T4 GPU*) for ultra-fast synthesis, or `CPU`.
- **Python Version:** Python 3 (Google Colab standard environment).


## 0. Workflow Overview

1. **Install Dependencies:** `p7zip-full`, `ffmpeg`, and Python libraries (`requests`, `soundfile`).
2. **GPU Detection & Engine Startup:** Detects NVIDIA GPU with `nvidia-smi`, downloads the official release binary (`voicevox_engine-linux-nvidia` or `voicevox_engine-linux-cpu`), launches the engine with `--use_gpu`, and validates the `http://127.0.0.1:50021/version` endpoint.
3. **Speaker Inspection & Speech Test:** Lists voices and synthesizes a test phrase.
4. **Presentation Batch Synthesis:** Reads canonical `slides_reading_ja_kana.txt` (continuous natural Japanese without artificial spaces), validates queries, applies global `prePhonemeLength = 0.5s` and `postPhonemeLength = 0.5s`, and packages MP3s into a ZIP.


In [ ]:
# ==============================================================================
# 1. Install System & Python Dependencies (Zero Docker)
# ==============================================================================

!apt-get update -qq
!apt-get install -y -qq p7zip-full wget curl ffmpeg > /dev/null

!pip install -q requests soundfile

print("System packages and Python dependencies installed successfully.")


In [ ]:
# ==============================================================================
# 2. Detect GPU, Download Official VOICEVOX Engine & Start Service
# ==============================================================================

import os
import subprocess
import time
from pathlib import Path
import requests

VERSION = "0.25.2"
HOST = "http://127.0.0.1:50021"
LOG_FILE = Path("voicevox_engine.log").resolve()

# 1. Run nvidia-smi and detect GPU
print("=" * 65)
print("[1/4] DETECTING HARDWARE ACCELERATOR (nvidia-smi)...")
print("=" * 65)

has_gpu = False
gpu_model = "Unknown"

try:
    smi_out = subprocess.check_output(["nvidia-smi"], text=True)
    print(smi_out.strip())
    has_gpu = True
    try:
        gpu_model = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
        ).strip()
    except Exception:
        gpu_model = "NVIDIA GPU (Detected)"
except (FileNotFoundError, subprocess.CalledProcessError):
    print("No NVIDIA GPU detected. Proceeding with CPU backend.")

# 2. Download and extract official release binary
print("\n" + "=" * 65)
if has_gpu:
    print(f"[2/4] DOWNLOADING OFFICIAL LINUX GPU/CUDA RELEASE (v{VERSION})...")
    print("=" * 65)
    engine_dir = Path("./linux-nvidia").resolve()
    if not (engine_dir / "run").exists():
        url_001 = f"https://github.com/VOICEVOX/voicevox_engine/releases/download/{VERSION}/voicevox_engine-linux-nvidia-{VERSION}.7z.001"
        url_002 = f"https://github.com/VOICEVOX/voicevox_engine/releases/download/{VERSION}/voicevox_engine-linux-nvidia-{VERSION}.7z.002"
        print(f"Fetching: {url_001}")
        subprocess.run(["wget", "-q", "-c", url_001], check=True)
        print(f"Fetching: {url_002}")
        subprocess.run(["wget", "-q", "-c", url_002], check=True)
        print("Extracting GPU archive with 7z...")
        subprocess.run(["7z", "x", "-y", f"voicevox_engine-linux-nvidia-{VERSION}.7z.001"], stdout=subprocess.DEVNULL, check=True)
    engine_binary = engine_dir / "run"
    run_args = [str(engine_binary), "--use_gpu", "--host", "127.0.0.1", "--port", "50021"]
else:
    print(f"[2/4] DOWNLOADING OFFICIAL LINUX CPU RELEASE (v{VERSION})...")
    print("=" * 65)
    engine_dir = Path("./linux-cpu-x64").resolve()
    if not (engine_dir / "run").exists():
        url_cpu = f"https://github.com/VOICEVOX/voicevox_engine/releases/download/{VERSION}/voicevox_engine-linux-cpu-x64-{VERSION}.7z.001"
        print(f"Fetching: {url_cpu}")
        subprocess.run(["wget", "-q", "-c", url_cpu], check=True)
        print("Extracting CPU archive with 7z...")
        subprocess.run(["7z", "x", "-y", f"voicevox_engine-linux-cpu-x64-{VERSION}.7z.001"], stdout=subprocess.DEVNULL, check=True)
    engine_binary = engine_dir / "run"
    run_args = [str(engine_binary), "--host", "127.0.0.1", "--port", "50021"]

# 3. Make binary executable and start process in background
print("\n" + "=" * 65)
print("[3/4] STARTING VOICEVOX ENGINE SERVICE...")
print("=" * 65)
os.chmod(engine_binary, 0o755)

subprocess.run(["pkill", "-f", "voicevox_engine"], stderr=subprocess.DEVNULL)
time.sleep(1)

log_out = open(LOG_FILE, "w", encoding="utf-8")
proc = subprocess.Popen(run_args, stdout=log_out, stderr=subprocess.STDOUT, cwd=str(engine_dir))
print(f"Engine spawned with PID {proc.pid}. Logging to: {LOG_FILE}")

# 4. Poll /version endpoint until online
print("\n" + "=" * 65)
print("[4/4] VERIFYING /version ENDPOINT...")
print("=" * 65)

online_version = None
for sec in range(90):
    try:
        resp = requests.get(f"{HOST}/version", timeout=2)
        if resp.status_code == 200:
            online_version = resp.text.strip().strip('"')
            break
    except Exception:
        if sec % 5 == 0:
            print(f"  Waiting for engine startup... ({sec}s/90s)")
    time.sleep(1)

if online_version:
    print("\n" + "=" * 65)
    if has_gpu:
        print("VOICEVOX BACKEND: GPU")
        print(f"GPU MODEL: {gpu_model}")
    else:
        print("VOICEVOX BACKEND: CPU")
    print(f"ENGINE VERSION: {online_version}")
    print(f"STATUS: ONLINE (http://127.0.0.1:50021)")
    print("=" * 65)
else:
    print("[ERROR] Engine failed to respond within 90s.")
    raise RuntimeError("VOICEVOX Engine did not start.")


In [ ]:
# ==============================================================================
# 3. List Available Speakers
# ==============================================================================

import requests

HOST = "http://127.0.0.1:50021"
speakers = requests.get(f"{HOST}/speakers").json()

print(f"Total voice models loaded: {len(speakers)}\n")
for speaker in speakers:
    print(speaker["name"])
    for style in speaker["styles"]:
        print(f"   ID: {style['id']:2d} | {style['name']}")


In [ ]:
# ==============================================================================
# 4. Generate Single Speech Test & Validate Query
# ==============================================================================

import json
import requests
from pathlib import Path
from IPython.display import Audio, display

HOST = "http://127.0.0.1:50021"
speaker_id = 13  # 青山龍星 (ノーマル)

sample_text = "これはにほんごのよみあげテストです。VOICEVOXエンジンがせいじょうにどうさしています。"

query = requests.post(
    f"{HOST}/audio_query",
    params={"text": sample_text, "speaker": speaker_id}
).json()

# Query validation
assert query.get("kana"), "Query validation failed: no kana produced"
print("TEXT SENT     :", repr(sample_text))
print("VOICEVOX KANA :", query.get("kana"))

query["speedScale"] = 0.95
query["pitchScale"] = 0.0
query["intonationScale"] = 1.0
query["volumeScale"] = 1.0
query["prePhonemeLength"] = 0.5
query["postPhonemeLength"] = 0.5

wav = requests.post(
    f"{HOST}/synthesis",
    params={"speaker": speaker_id},
    data=json.dumps(query),
    headers={"Content-Type": "application/json"}
).content

test_out = Path("voicevox_test.wav")
test_out.write_bytes(wav)

print("\nSample test audio generated successfully:")
display(Audio(str(test_out)))


In [ ]:
# ==============================================================================
# 5. Production Presentation Audio Synthesis (Global 0.5s Pre/Post Roll Padding)
# ==============================================================================
# - Standardizes prePhonemeLength = 0.5s and postPhonemeLength = 0.5s across all slides.
# - Clean, natural neural synthesis with query validation and fast resume logic.
# - Supports TEST_MODE for rapid sample verification before full production run.
# ==============================================================================

import json
import re
import requests
import subprocess
import zipfile
from pathlib import Path

HOST = "http://127.0.0.1:50021"

# ------------------------------------------------------------------------------
# 1. HEALTH CHECK
# ------------------------------------------------------------------------------
try:
    health_resp = requests.get(f"{HOST}/version", timeout=5)
    health_resp.raise_for_status()
    print(f"[OK] VOICEVOX Engine verified online (v{health_resp.text.strip()}).")
except Exception:
    raise SystemExit("[ERROR] VOICEVOX Engine is not reachable at http://127.0.0.1:50021. Please run Cell 2 first.")

# ------------------------------------------------------------------------------
# 2. CONFIGURATION
# ------------------------------------------------------------------------------
SPEAKER_ID = 13
OUTPUT_DIR = Path("voicevox_mp3_presentation_ja")
OUTPUT_DIR.mkdir(exist_ok=True)

# Test Mode: synthesize only representative sample slides before full batch
TEST_MODE = False
SAMPLE_SLIDE_INDICES = [1, 2, 10, 30]

PRE_PHONEME_LENGTH = 0.50
POST_PHONEME_LENGTH = 0.50

MP3_BITRATE = "128k"
SPEED_SCALE = 0.95
PITCH_SCALE = 0.0
INTONATION_SCALE = 1.0
VOLUME_SCALE = 1.0

FORCE_REGENERATE = set()

# ------------------------------------------------------------------------------
# 3. LOAD CANONICAL KANA LAYER
# ------------------------------------------------------------------------------
kana_file = Path("slides_reading_ja_kana.txt")

if kana_file.exists():
    print(f"[INPUT] Loading canonical Japanese Kana layer from: {kana_file.resolve()}")
    content = kana_file.read_text(encoding="utf-8")
    raw_slides = re.split(r"(?:スライド)\s+(\d+)", content)
    parsed_dict = {}
    for i in range(1, len(raw_slides), 2):
        s_num = int(raw_slides[i])
        parsed_dict[s_num] = raw_slides[i+1].strip()

    slide_texts = []
    for i in range(1, len(parsed_dict) + 1):
        tb = parsed_dict[i]
        if i == 1:
            title, subtitle, author = "", "", ""
            for l in tb.split("\n"):
                if l.startswith("タイトル:"): title = l.replace("タイトル:", "").strip().replace(" ", "").replace("　", "")
                elif l.startswith("サブタイトル:"): subtitle = l.replace("サブタイトル:", "").strip().replace(" ", "").replace("　", "")
                elif l.startswith("著者:"): author = l.replace("著者:", "").strip().replace(" ", "").replace("　", "")
            spoken = f"{title}。{subtitle}。{author}。"
        elif 2 <= i <= len(parsed_dict) - 2:
            spoken = ""
            for l in tb.split("\n"):
                if l.startswith("本文:"):
                    spoken = l.replace("本文:", "").strip().replace(" ", "").replace("　", "")
                    break
        else:
            title = ""
            words = []
            for l in tb.split("\n"):
                l = l.strip()
                if l.startswith("タイトル:"): title = l.replace("タイトル:", "").strip().replace(" ", "").replace("　", "")
                elif re.match(r"^\d+\.\s*", l):
                    item = re.sub(r"^\d+\.\s*", "", l).strip()
                    item_clean = item.replace("—", "、").replace(" ", "").replace("　", "").strip()
                    words.append(item_clean)
            spoken = f"{title}。" + "。".join(words) + "。"
        slide_texts.append(spoken)
else:
    print("[INPUT] 'slides_reading_ja_kana.txt' not found. Using embedded fallback.")
    slide_texts = [
        "にほんのなつやすみ。でんとう、おまつり、そしてはれやかなおもいで。ファビオレイバ、エフエル。",
        "にほんのなつやすみは、しちがつげじゅんにはじまります。",
        "まいあさ、きぎのなかでセミがにぎやかにないています。",
        "せいとたちはともだちとたのしいけいかくをたてます。",
        "せんせいはなつやすみようのしゅくだいのワークブックをくばります。",
        "こどもたちはあさのラジオたいそうのために、きんじょのこうえんにあつまります。",
        "まいあさ、たいそうカードにカラフルなスタンプをおしてもらいます。",
        "おおくのしょうがくせいがいえであさがおのみずやりをします。",
        "こどもたちはえにっきになつのおもいでをかいたりえがいたりします。",
        "こどもたちはもりでカブトムシをつかまえるためにはやおきします。",
        "ともだちといっしょにしみんプールでおよいだりみずあそびをしたりします。",
        "こどもたちはこうえんでたのしいみずでっぽうあそびをします。",
        "ともだちといっしょにきのぼうでスイカをたたくスイカわりにちょうせんします。",
        "ふわふわのかきごおりにあまいイチゴシロップをかけます。",
        "つめたいそうめんがながいたけのといをながれていきます。",
        "ビーだまをおしこんで、キンキンにひえたラムネをのみます。",
        "やたいではあつあつのやきそばややきとうもろこしがうられています。",
        "あついなつのひにスタミナをつけるため、うなぎのかばやきをたべます。",
        "かぞくみんなでスーツケースになつふくやぼうしをつめこみます。",
        "しんかんせんがりょこうしゃをのせてでんえんちたいをはしりぬけます。",
        "ひこうきがかぞくづれをみなみのたいようがかがやくしまじまへはこびます。",
        "うみぞいのこうそくどうろをはしりながらかぞくでドライブをたのしみます。",
        "すなはまではいろとりどりのパラソルのしたでかいすいよくきゃくがくつろいでいます。",
        "すいちゅうメガネをつけて、うみのなかをおよぐいろあざやかなさかなをかんさつします。",
        "こどもたちはおおきなすなのしろをつくったり、きれいなかいがらをひろったりします。",
        "うみのいえではつめたいのみものやひかげのきゅうけいスペースがよういされています。",
        "あかりのともったちょうちんがにぎやかなおまつりのとおりをてらします。",
        "いろあざやかなめんのゆかたをきて、きのげたをはいてでかけます。",
        "えんにちのやたいではたのしいけいひんがもらえるゲームでもりあがります。",
        "こどもたちはやぶれやすいポイをつかってきんぎょすくいにちょうせんします。",
        "おおきなたいこのおとにあわせて、ひとびとがやぐらのまわりをわになっておどります。",
        "おおぜいのひとがかせんじきにすわってはなびたいかいをたのしみます。",
        "よぞらいっぱいにきょだいでいろあざやかなはなびがうちあがります。",
        "こどもたちはしずかなにわでキラキラとかがやくせんこうはなびをたのしみます。",
        "せんこうはなびをみつめるじかんは、こころにのこるなつのおもいでになります。",
        "すんだくうきとゆたかなしぜんをもとめて、かぞくでしずかないなかへでかけます。",
        "いちめんにひろがるあおあおとしたたんぼがたにまにつづいています。",
        "すんだおがわのつめたいみずでスイカやのみものをひやします。",
        "ゆうぐれどきになると、みずべをホタルがやさしくひかりながらまいます。",
        "おぼんはせんぞのれいをおむかえしてくようするはちがつのたいせつなぎょうじです。",
        "かぞくみんなでさとがえりし、おじいちゃんやおばあちゃんにあいにいきます。",
        "おはかをきれいにそうじして、おはなやおせんこうをおそなえします。",
        "とうろうながしのあかりが、せんぞのれいをかわへとやさしくおくりだします。",
        "せんぷうきのかぜをかんじながら、すずしいたたみのへやでかぞくとくつろぎます。",
        "えんがわにすわって、みんなでつめたくてあまいスイカをたべます。",
        "よるにはかぞくみんなでテーブルをかこんでボードゲームをたのしみます。",
        "こどもたちはいえでこうさくやじゆうけんきゅうにとりくみます。",
        "はちがつげじゅんのこがねいろのゆうぐれとともに、すずしいゆうかぜがふきはじめます。",
        "しんがっきがはじまるまえに、のこっているなつやすみのしゅくだいをおわらせます。",
        "たくさんのしゃしんととくべつなおもいでは、いつまでもこころにのこります。",
        "くがつにクラスメイトとさいかいできるのを、せいとたちはたのしみにしています。",
        "じゅうようたんごいち。きゅうか、がっこうやしごとがやすみのきかん。おまつり、とくべつなおいわいのぎょうじ。はなび、よぞらをいろどるひかり。でんとう、むかしからうけつがれてきたふうしゅう。おもいで、こころにのこるとくべつなきおく。こきょう、うまれそだったばしょやじっかのあるまち。さわやか、すっきりしてきもちがよいこと。おいしい、あじがとてもよいこと。こんちゅう、ろっぽんあしのちいさなむし。スーツケース、りょこうのにもつをいれるかばん。",
        "じゅうようたんごに。ちょうちん、かみなどでかこまれたあかり。いなか、とかいからはなれたちいき。せんぞ、じぶんよりまえのせだいのかぞく。やたい、たべものやものをうるちいさなおみせ。おがわ、みずがながれるちいさなかわ。いしょう、おまつりやぎょうじのためのとくべつなふく。にもつ、りょこうにもっていくかばんなど。げんきな、エネルギーにあふれていること。せんこうはなび、てにもってたのしむちいさなはなび。リラックスした、こころがおちついていること。",
    ]

print(f"[VALIDATION] Total slide entries loaded: {len(slide_texts)}")

# ------------------------------------------------------------------------------
# 4. SYNTHESIS FUNCTION (CLEAN NATIVE PRE/POST PADDING + QUERY VALIDATION)
# ------------------------------------------------------------------------------
def synthesize_slide_mp3(text, out_mp3_path):
    query = requests.post(
        f"{HOST}/audio_query",
        params={"text": text, "speaker": SPEAKER_ID},
        timeout=60
    ).json()

    if not query.get("kana"):
        raise ValueError(f"Query validation failed for text: {text}")

    query["speedScale"] = SPEED_SCALE
    query["pitchScale"] = PITCH_SCALE
    query["intonationScale"] = INTONATION_SCALE
    query["volumeScale"] = VOLUME_SCALE
    query["prePhonemeLength"] = PRE_PHONEME_LENGTH
    query["postPhonemeLength"] = POST_PHONEME_LENGTH

    wav_bytes = requests.post(
        f"{HOST}/synthesis",
        params={"speaker": SPEAKER_ID},
        data=json.dumps(query),
        headers={"Content-Type": "application/json"},
        timeout=120
    ).content

    command = [
        "ffmpeg",
        "-y",
        "-hide_banner",
        "-loglevel", "error",
        "-i", "pipe:0",
        "-vn",
        "-codec:a", "libmp3lame",
        "-b:a", MP3_BITRATE,
        str(out_mp3_path)
    ]
    res = subprocess.run(command, input=wav_bytes, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if res.returncode != 0:
        raise RuntimeError(res.stderr.decode("utf-8", errors="replace"))

# ------------------------------------------------------------------------------
# 5. PRODUCTION BATCH SYNTHESIS
# ------------------------------------------------------------------------------
target_indices = SAMPLE_SLIDE_INDICES if TEST_MODE else list(range(1, len(slide_texts) + 1))
print(f"\n{'='*65}")
print(f"SYNTHESIZING PRESENTATION AUDIO ({len(target_indices)} slides to process)...")
print(f"{'='*65}")

generated_count = 0
skipped_count = 0
all_mp3_paths = []

for idx in target_indices:
    text = slide_texts[idx - 1]
    out_mp3 = OUTPUT_DIR / f"slide_{idx:02d}.mp3"
    all_mp3_paths.append(out_mp3)

    if out_mp3.exists() and out_mp3.stat().st_size > 0 and idx not in FORCE_REGENERATE:
        skipped_count += 1
        continue

    print(f"--> [{idx:02d}/{len(slide_texts)}] Synthesizing {out_mp3.name}: '{text[:45]}...'")
    synthesize_slide_mp3(text, out_mp3)
    generated_count += 1

print(f"\n[SUMMARY] Synthesis finished: {generated_count} generated, {skipped_count} skipped.")

# ------------------------------------------------------------------------------
# 6. VERIFY & ZIP PACKAGING (NON-TEST MODE)
# ------------------------------------------------------------------------------
if not TEST_MODE:
    missing_or_empty = [p.name for p in all_mp3_paths if not p.exists() or p.stat().st_size == 0]
    if missing_or_empty:
        raise RuntimeError(f"Missing or empty audio files: {missing_or_empty}")

    zip_path = Path("presentation_ja_audio_mp3.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in all_mp3_paths:
            zf.write(p, arcname=p.name)

    print(f"\n=================================================================")
    print(f"★ PRODUCTION AUDIO BATCH READY: {len(all_mp3_paths)} tracks verified!")
    print(f"★ ZIP Archive Created: {zip_path.resolve()} ({zip_path.stat().st_size / 1024:.1f} KB)")
    print(f"=================================================================")
